In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import os

In [21]:
X_train = pd.read_csv("../data/imbalanced/X_SMOTE.csv")
y_train = pd.read_csv("../data/imbalanced/y_SMOTE.csv").values.ravel()

print("SMOTE Train Shape:", X_train.shape)
print("SMOTE Labels Shape:", y_train.shape)

SMOTE Train Shape: (1768, 11)
SMOTE Labels Shape: (1768,)


In [22]:
X_test = pd.read_csv("../data/processed/X_test_processed.csv")

y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Test Shape:", X_test.shape)

Test Shape: (333, 11)


In [23]:
normalization_methods = {

    "MinMaxScaler":
        MinMaxScaler(),

    "StandardScaler":
        StandardScaler()
}

In [24]:
SAVE_PATH = "../data/normalized/"

os.makedirs(SAVE_PATH, exist_ok=True)

In [25]:
for name, scaler in normalization_methods.items():

    print(f"\n🚀 Applying: {name}")

    # Fit scaler ONLY on train
    X_train_scaled = scaler.fit_transform(X_train)

    # Transform test
    X_test_scaled = scaler.transform(X_test)

    # Save train
    pd.DataFrame(X_train_scaled).to_csv(
        SAVE_PATH + f"X_train_{name}.csv",
        index=False
    )

    # Save test
    pd.DataFrame(X_test_scaled).to_csv(
        SAVE_PATH + f"X_test_{name}.csv",
        index=False
    )

    print("✅ Saved")


🚀 Applying: MinMaxScaler
✅ Saved

🚀 Applying: StandardScaler
✅ Saved


In [26]:
results = []

for name in normalization_methods.keys():

    print(f"\n📊 Evaluating: {name}")

    # Load normalized train data
    X_train_scaled = pd.read_csv(
        SAVE_PATH + f"X_train_{name}.csv"
    )

    # Load normalized test data
    X_test_scaled = pd.read_csv(
        SAVE_PATH + f"X_test_{name}.csv"
    )

    # Train model
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Metrics
    acc = accuracy_score(y_test, y_pred)

    prec = precision_score(
        y_test,
        y_pred,
        average="macro"
    )

    rec = recall_score(
        y_test,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    results.append({

        "Normalization": name,

        "Accuracy": round(acc, 4),

        "Precision": round(prec, 4),

        "Recall": round(rec, 4),

        "F1-Score": round(f1, 4)
    })

    print("✅ Done")


📊 Evaluating: MinMaxScaler
✅ Done

📊 Evaluating: StandardScaler
✅ Done


In [27]:
normalization_results_df = pd.DataFrame(results)

normalization_results_df = normalization_results_df.sort_values(
    by="F1-Score",
    ascending=False
)

print("\n📊 NORMALIZATION RESULTS\n")

print(normalization_results_df)


📊 NORMALIZATION RESULTS

    Normalization  Accuracy  Precision  Recall  F1-Score
0    MinMaxScaler       1.0        1.0     1.0       1.0
1  StandardScaler       1.0        1.0     1.0       1.0


In [28]:
os.makedirs("../reports", exist_ok=True)

normalization_results_df.to_csv(
    "../reports/normalization_results.csv",
    index=False
)

print("✅ Results Saved")

✅ Results Saved
